# W12B · 하이브리드 추천 시스템 · 두 예측을 하나의 추천으로

동덕여자대학교 데이터사이언스전공 · 유원상 · 딥러닝응용I(추천시스템) 01 · 2026-2 · 11월 19일

주교재 10.1–10.3과 연습 10-1. 강의노트의 S01–S06 순서로 영화 A와 B를 따라갑니다. 값을 예상하고 표시된 셀만 고친 뒤 다시 실행하세요. A와 B는 가상 영화이고 S04는 MovieLens 100K의 실제 평점입니다.


## S01 · 같은 실제 평점, 두 예측의 방향

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f01-errors.png" alt="실제값에서 두 예측 오차의 방향" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f01-errors.png)

Q01. 실제 4점, CF 3점, MF 5점에서 오차의 부호와 50:50 결합을 예상하세요. MF가 2점이면 어떻게 달라질까요?


In [ ]:
from math import sqrt
from IPython.display import display

movie_rows = [
    {"user_id":"관객","movie_id":"A","rating":4.0,"score_cf":3.0,"score_mf":5.0,"score_mlp":3.8},
    {"user_id":"관객","movie_id":"B","rating":4.0,"score_cf":4.5,"score_mf":3.5,"score_mlp":4.1},
]
for example, scores in (("반대 방향",(3,5)),("같은 방향",(3,2))):
    print(example,"오차",[p-4 for p in scores],"동일 비중",sum(scores)/2)


## S02 · 결합식과 RMSE를 한 줄씩 읽기

MF 비중 $\alpha$에 대해 $\hat r_H=(1-\alpha)\hat r_{CF}+\alpha\hat r_{MF}$입니다. RMSE는 개별 행 오차의 제곱평균에 제곱근을 취합니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f02-components.png" alt="영화별 CF와 MF의 가중 기여" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f02-components.png)

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f03-rmse.png" alt="두 행의 제곱오차와 RMSE" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f03-rmse.png)

Q02. 먼저 비중 0.8의 점수와 RMSE를 적습니다. 함수 안에서 모형마다 어떤 항이 곱해지는지 찾아 표시하고 호출을 0.5로 바꿉니다.


In [ ]:
def blend_one(score_cf,score_mf,weight_mf):
    if not 0<=weight_mf<=1:
        raise ValueError("MF 비중은 0과 1 사이여야 합니다.")
    cf_part=(1-weight_mf)*score_cf
    mf_part=weight_mf*score_mf
    return cf_part,mf_part,cf_part+mf_part

def rmse(labels,predictions):
    labels,predictions=list(labels),list(predictions)
    if not labels or len(labels)!=len(predictions):
        raise ValueError("비어 있지 않은 실제값과 예측값의 행을 맞추세요.")
    squared=[(pred-label)**2 for label,pred in zip(labels,predictions)]
    return sqrt(sum(squared)/len(squared))

def toy_report(rows,weight_mf):
    scores=[blend_one(row["score_cf"],row["score_mf"],weight_mf) for row in rows]
    for row,(cf_part,mf_part,total) in zip(rows,scores):
        print(row["movie_id"],f"기여 {cf_part:.2f} + {mf_part:.2f} = {total:.2f}",
              f"제곱오차 {(total-row['rating'])**2:.4f}")
    loss=rmse((row["rating"] for row in rows),(s[-1] for s in scores))
    print("평가 행",len(rows),"RMSE",round(loss,6))
    return loss

toy_rmse=toy_report(movie_rows,.8)


## S03 · CF와 MF가 한 행의 점수를 만드는 방법

Q03. CF의 대상 평균 3.5, 이웃 편차 −1.5와 +1.5, 가중치 2/3와 1/3을 계산하세요. MF는 전체평균 3.5, 편향 0.2와 0.3, 두 $(0.8,0.6)$ 벡터의 내적을 더합니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f04-cf.png" alt="대상 평균과 이웃 편차가 만든 CF" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f04-cf.png)

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f05-mf.png" alt="평균 편향 내적이 만든 MF" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f05-mf.png)


In [ ]:
def cf_raw(mean,deviations,weights):
    return mean+sum(d*w for d,w in zip(deviations,weights))

def mf_raw(mean,user_bias,movie_bias,p,q):
    return mean+user_bias+movie_bias+sum(x*y for x,y in zip(p,q))

print("CF",cf_raw(3.5,[-1.5,1.5],[2/3,1/3]))
print("MF",mf_raw(3.5,.2,.3,[.8,.6],[.8,.6]))


### S03 · 두 예측표에서 같은 사용자와 영화 찾기

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f06-keys.png" alt="섞인 두 표를 사용자와 영화 키로 연결" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f06-keys.png)

Q04. CF 순서 [A,B], MF 순서 [B,A]입니다. 위치로 붙이면 A가 몇 점인지 예상합니다. 함수에서 두 키를 함께 쓰는 줄을 찾고, 호출에서 MF 한 행을 빼서 진단 결과를 읽으세요.


In [ ]:
import numpy as np
import pandas as pd

PAIR=["user_id","movie_id"]

def align_predictions(labels,first,second):
    # 수정할 핵심: 두 열을 묶어 같은 사용자와 영화를 찾는다.
    join_keys=["user_id","movie_id"]
    for table in [labels,first,second]:
        if table[join_keys].isna().any().any() or table.duplicated(join_keys).any():
            raise ValueError("키에 빈 값 또는 중복이 있습니다.")
    expected=set(labels[join_keys].itertuples(index=False,name=None))
    for table in [first,second]:
        actual=set(table[join_keys].itertuples(index=False,name=None))
        if actual!=expected:
            raise ValueError(f"누락 {len(expected-actual)}행 / 초과 {len(actual-expected)}행")
    joined=labels.merge(first,on=join_keys,how="left",validate="one_to_one")
    return joined.merge(second,on=join_keys,how="left",validate="one_to_one")

small=pd.DataFrame(movie_rows)
cf_small=small[PAIR+["score_cf"]]
mf_small=small[PAIR+["score_mf"]].iloc[::-1]
small_labels=small[PAIR+["rating"]]
print("CF 첫 행",cf_small.iloc[0].to_dict())
print("MF 첫 행",mf_small.iloc[0].to_dict())
print("위치로 붙인 첫 점수",blend_one(cf_small.iloc[0].score_cf,mf_small.iloc[0].score_mf,.8)[-1])
# 누락 실험은 mf_small 대신 mf_small.iloc[:1]을 사용한다.
aligned_small=align_predictions(small_labels,cf_small,mf_small)
display(aligned_small)


## S04 · 실제 MovieLens와 시간순 분할

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f07-time.png" alt="훈련 검증 시험을 시간순으로 나누기" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f07-time.png)

처음에는 GroupLens 배포처에서 평점을 내려받습니다. 네트워크가 지연되면 RUN_REAL을 False로 바꿔 합성 표로 검증·시험 흐름을 계속 진행할 수 있습니다. 실제 실험의 지원 부족에는 두 모델에 공통 훈련 평균을 쓰고 1–5점 범위를 지킵니다.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from time import perf_counter
import matplotlib.pyplot as plt

os.environ.update(CUDA_VISIBLE_DEVICES="-1",TF_CPP_MIN_LOG_LEVEL="2",TF_ENABLE_ONEDNN_OPTS="0")
RUN_REAL=True
RUN_MLP=os.environ.get("LUNA_W12B_RUN_MLP")=="1"
IN_COLAB="google.colab" in sys.modules
MODEL_REV="d6f387299f8ab6aa307a1c66578e3c5a5e82bbf0"
default_cache=Path("data/local") if not IN_COLAB and Path("data/local/ml-100k/u.data").is_file() else Path.home()/".cache/luna-recsys/w12b"
CACHE=Path(os.environ.get("LUNA_W12B_CACHE",default_cache))

def temporal_split(ratings):
    keys=ratings[PAIR+["timestamp","rating"]]
    if keys.isna().any().any() or keys.duplicated(PAIR).any() or not keys.rating.between(1,5).all():
        raise ValueError("키·시각·평점의 누락과 척도를 확인하세요.")
    rows=ratings.sort_values(["timestamp",*PAIR]).reset_index(drop=True)
    first=rows.timestamp.iloc[int(.70*len(rows))]
    second=rows.timestamp.iloc[int(.85*len(rows))]
    parts={"train":rows.loc[rows.timestamp<first],
           "validation":rows.loc[(rows.timestamp>=first)&(rows.timestamp<second)],
           "test":rows.loc[rows.timestamp>=second]}
    if any(part.empty for part in parts.values()):
        raise ValueError("세 구간에 모두 관측이 필요합니다.")
    return parts

def predict_pair_tables(cf,mf,pairs):
    keys=pairs[PAIR].reset_index(drop=True)
    evidence=cf.predict_details(keys)
    cf_supported=evidence.basis.eq("cf").to_numpy()
    cf_score=np.where(cf_supported,evidence.prediction,cf.global_mean_)
    u=keys.user_id.map(mf.user_index_)
    i=keys.movie_id.map(mf.item_index_)
    mf_supported=(u.notna()&i.notna()).to_numpy()
    ui=u[mf_supported].to_numpy(int)
    ii=i[mf_supported].to_numpy(int)
    mf_score=np.full(len(keys),cf.global_mean_)
    mf_score[mf_supported]=(mf.mean_+mf.user_bias_[ui]+mf.item_bias_[ii]
                            +(mf.P_[ui]*mf.Q_[ii]).sum(axis=1))
    return (keys.assign(score_cf=np.clip(cf_score,1,5),supported_cf=cf_supported),
            keys.assign(score_mf=np.clip(mf_score,1,5),supported_mf=mf_supported))

def trace_pair(cf,mf,user_id,movie_id):
    pair=pd.DataFrame([(user_id,movie_id)],columns=PAIR)
    cf_table,mf_table=predict_pair_tables(cf,mf,pair)
    evidence=cf.explain(user_id,movie_id)
    if not evidence.empty:
        base=float(cf.user_means_.loc[user_id])
        total=cf_raw(base,evidence.value,evidence.weight)
        assert np.isclose(np.clip(total,1,5),cf_table.score_cf.iloc[0])
        display(evidence[["neighbor_id","rating","reference_mean","similarity","value","weight","contribution"]])
        print(f"CF 대상 평균 {base:.4f} + 이웃 기여 {evidence.contribution.sum():.4f}")
    else:
        print("CF: 근거 부족, 훈련 전체 평균 대체")
    if mf_table.supported_mf.iloc[0]:
        u,i=mf.user_index_[user_id],mf.item_index_[movie_id]
        visit=mf.explain(user_id,movie_id)
        total=mf_raw(mf.mean_,mf.user_bias_[u],mf.item_bias_[i],mf.P_[u],mf.Q_[i])
        assert np.isclose(np.clip(total,1,5),mf_table.score_mf.iloc[0])
        display(pd.Series({n:visit[n] for n in ("mean","user_bias","item_bias","interaction","prediction")}))
        display(pd.DataFrame({"p":mf.P_[u],"q":mf.Q_[i],"곱":visit["contributions"]}))
    else:
        print("MF: 미관측 ID, 훈련 전체 평균 대체")
    return cf_table.score_cf.iloc[0],mf_table.score_mf.iloc[0]

start=perf_counter()
if RUN_REAL:
    if IN_COLAB:
        subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps",
                        f"git+https://github.com/lunalab-ai/recommender.git@{MODEL_REV}"],check=True)
    from luna_recsys.cf_synthesis import EvidenceCF
    from luna_recsys.data import download_movielens_100k,load_movielens_100k
    from luna_recsys.mf_sgd import MFSGD
    movie_data=load_movielens_100k(download_movielens_100k(CACHE/"movielens"),expect_full=True)
    parts=temporal_split(movie_data.ratings)
    train=parts["train"]
    cf=EvidenceCF(k=37,min_common=3,min_neighbors=2,centered=True).fit(train)
    mf=MFSGD(n_factors=12,epochs=12,learning_rate=.02,regularization=.035,seed=20261117).fit(train)
    valid_labels=parts["validation"][PAIR+["rating"]]
    cf_valid,mf_valid=predict_pair_tables(cf,mf,valid_labels[PAIR])
    valid=align_predictions(valid_labels,cf_valid,mf_valid.sample(frac=1,random_state=19))
    supported=valid.loc[valid.supported_cf&valid.supported_mf].iloc[0]
    traced=trace_pair(cf,mf,int(supported.user_id),int(supported.movie_id))
    print("MovieLens 100K",{name:len(part) for name,part in parts.items()},"준비 초",round(perf_counter()-start,2))
    print("모델 지원율",valid[["supported_cf","supported_mf"]].mean().to_dict())
    print("실제 한 행의 CF와 MF",np.round(traced,5))
else:
    valid=aligned_small.assign(supported_cf=True,supported_mf=True)
    print("가상 2행 검증 표로 실습")


### S04 · 대체 전 지원율을 읽기

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f08-coverage.png" alt="대체 전 지원율과 동일한 전체 평가 분모" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f08-coverage.png)

Q05. 실제 다섯 행에서 CF가 세 행을 지원합니다. 한 미지원 행은 실제 5점, 훈련 전체 평균으로 대체한 두 점수는 3.6점입니다. 지원율과 이 행의 제곱오차를 구하세요.


### S04 · 검증 후보 중 하나를 선택해 고정하기

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f09-grid.png" alt="가상 검증 곡선과 이전 W12A의 실제 곡선" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f09-grid.png)

Q06. 촘촘한 0.01 간격을 0.25로 바꾸세요. 목록과 선택값이 어떻게 달라지는지 기록하고, 비중을 고정한 뒤 시험 셀을 실행하세요.


In [ ]:
def blend_scores(frame,weight_mf):
    cf_scores=frame.score_cf.to_numpy(float)
    mf_scores=frame.score_mf.to_numpy(float)
    if not (np.isfinite(cf_scores).all() and np.isfinite(mf_scores).all()):
        raise ValueError("각 행에는 두 개의 유한한 예측이 필요합니다.")
    return blend_one(cf_scores,mf_scores,weight_mf)[-1]

def select_weight(validation,grid=None):
    # 수정할 핵심: 101을 5로 바꿔 검증 후보 간격과 선택값을 비교한다.
    candidates=np.linspace(0,1,101) if grid is None else grid
    losses=[(float(w),rmse(validation.rating,blend_scores(validation,w))) for w in candidates]
    curve=pd.DataFrame(losses,columns=["weight_mf","validation_rmse"])
    best=curve.sort_values(["validation_rmse","weight_mf"]).iloc[0]
    return float(best.weight_mf),curve

weight,curve=select_weight(valid)
print("검증으로 선택한 MF 비중",weight,"후보 개수",len(curve))
ax=curve.plot(x="weight_mf",y="validation_rmse",legend=False,color="#1573a5",figsize=(7,3))
ax.axvline(weight,color="#b85223",ls="--")
ax.set(xlabel="MF weight",ylabel="Validation RMSE",
       title="MovieLens 100K / validation" if RUN_REAL else "Synthetic / two validation rows")
plt.show()


### S04 · 독립 시험 평가

검증 결과를 확인하고 선택을 기록했으면 이 셀에서 실제 시험 평점을 사용합니다. 네 설정의 분모는 동일한 행 수입니다.


In [ ]:
def compare_on_test(test,weight,other_name="CF"):
    settings=[(other_name,0),("교재 출발값",.8),("MF",1),("검증 선택",weight)]
    return pd.DataFrame([(name,w,rmse(test.rating,blend_scores(test,w)),len(test))
                         for name,w in settings],columns=["모형","MF 비중","시험 RMSE","평가 행"])

if RUN_REAL:
    test_labels=parts["test"][PAIR+["rating"]]
    cf_test,mf_test=predict_pair_tables(cf,mf,test_labels[PAIR])
    test=align_predictions(test_labels,cf_test,mf_test)
else:
    test=pd.DataFrame([("다른 관객","C",4.,3.,4.5),("다른 관객","D",3.,3.5,3.25)],
                      columns=PAIR+["rating","score_cf","score_mf"])
test_results=compare_on_test(test,weight)
display(test_results)


## S05 · 순위 교차와 MF·MLP 결합

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f10-cross.png" alt="비중에 따라 영화 A와 B의 순위가 바뀌는 지점" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f10-cross.png)

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f11-mlp.png" alt="CF 점수 열을 MLP로 바꾸고 검증부터 반복" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f11-mlp.png)

Q07. $3+2\alpha=4.5-\alpha$를 푼 뒤 0.4와 0.6에서 누가 1위인지 예측합니다. 점수 열을 CF로 바꿔 보며 MF 비중을 어느 자료에서 다시 선택해야 하는지 설명하세요.


In [ ]:
def rank_candidates(rows,weight_mf):
    scored=rows.assign(cf_part=(1-weight_mf)*rows.score_cf,
                       mf_part=weight_mf*rows.score_mf,
                       hybrid=blend_scores(rows,weight_mf))
    ordered=scored.sort_values(["hybrid","movie_id"],ascending=[False,True]).reset_index(drop=True)
    return ordered.assign(rank=np.arange(1,len(ordered)+1))

for candidate_weight in (.4,.6):
    display(rank_candidates(aligned_small,candidate_weight)[["movie_id","cf_part","mf_part","hybrid","rank"]])

# 수정할 핵심: score_cf와 score_mlp를 번갈아 사용한다.
first_predictor="score_mlp"
toy_deep=small[PAIR+["rating",first_predictor,"score_mf"]].rename(columns={first_predictor:"score_cf"})
deep_weight,deep_curve=select_weight(toy_deep)
print("가상 MLP 검증 선택",deep_weight,"영화 A의 0.8 결합",blend_scores(toy_deep,.8)[0])


### S05 · 실제 MLP 재현

같은 훈련 사용자와 영화 ID를 임베딩하고 두 은닉층으로 중심화된 평점을 학습합니다. 실제 MLP 학습을 실행할 때만 이 셀에서 설정을 True로 변경합니다. 위의 가상 점수 교체 연습과 같은 키와 절차를 사용합니다.


In [ ]:
# 실습 환경처럼 전체 재현하려면 RUN_MLP를 True로 바꾼다.
if RUN_REAL and RUN_MLP:
    if IN_COLAB:
        subprocess.run([sys.executable,"-m","pip","install","-q",
                        "tensorflow==2.21.0","keras==3.15.1"],check=True)
    from luna_recsys.hybrid_lab import fit_deep_scores
    dl_valid,dl_test=fit_deep_scores(train,valid[PAIR],test[PAIR])
    dl_v=align_predictions(valid[PAIR+["rating"]],dl_valid,mf_valid)
    dl_t=align_predictions(test[PAIR+["rating"]],dl_test,mf_test)
    real_mlp_weight,real_mlp_curve=select_weight(dl_v)
    display(compare_on_test(dl_t,real_mlp_weight,other_name="MLP"))
    print("W12B_MLP_EXECUTED",len(dl_t),round(real_mlp_weight,2))
else:
    print("핵심 수업 경로: 위 합성 점수 열에서 10-1 비교 설계를 실행했습니다.")


### S05 · MF 갱신 한 걸음

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f12-sgd.png" alt="같은 갱신 전 잠재 요인으로 두 모수 갱신" width="1050">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w12b/course/notion/assets/w12b/f12-sgd.png)

Q08. $p=0.5$, $q=1.0$, 오차 0.5, 학습률 0.1일 때 두 새 값을 구하고, 갱신 전의 두 값을 쓰는 이유를 설명하세요.


In [ ]:
def one_step(p,q,rate,error,penalty=0):
    old_p,old_q=p,q
    p=old_p+rate*(error*old_q-penalty*old_p)
    q=old_q+rate*(error*old_p-penalty*old_q)
    return p,q

print("SGD example",one_step(.5,1.,.1,.5))


## S06 · 3분 확인

1. 두 예측에서 같은 행을 찾는 두 열은 무엇인가요?
2. 결합 비중과 최종 성능은 각각 어떤 분할에 의해 결정되나요?
3. 지원율의 분모와 시험 RMSE의 분모는 무엇을 세나요?
4. 결합의 비중이 0, 1일 때 어떤 모형이 남나요?
5. 순위 교차와 RMSE 감소를 각각 무엇으로 판단하나요?


In [ ]:
assert np.isclose(toy_rmse,sqrt(.225))
assert np.allclose(blend_scores(aligned_small,0),aligned_small.score_cf)
assert np.allclose(blend_scores(aligned_small,1),aligned_small.score_mf)
assert len(test_results)==4 and test_results["평가 행"].eq(len(test)).all()
assert set(test_results["MF 비중"]).issubset({0,.8,1,weight})
assert rank_candidates(aligned_small,.4).movie_id.iloc[0]!=rank_candidates(aligned_small,.6).movie_id.iloc[0]
if RUN_REAL:
    assert sum(map(len,parts.values()))==100000
    assert parts["train"].timestamp.max()<parts["validation"].timestamp.min()
    assert parts["validation"].timestamp.max()<parts["test"].timestamp.min()
    assert len(valid)>14000 and len(test)>14000
    assert np.allclose(traced,[supported.score_cf,supported.score_mf])
    if RUN_MLP:
        assert len(dl_t)==len(test) and dl_t.score_cf.between(1,5).all()
    print("W12B_REAL_RUNTIME_PASSED",len(valid),len(test),round(weight,2))
else:
    print("W12B_SYNTHETIC_RUNTIME_PASSED")


## 참고 · 모델 내부를 읽을 때 펼치기

CF 이웃 선택과 MF 내적, SGD 갱신에서 실제 사용한 모델 리비전의 소스 부분입니다. S03과 S05의 편집 가능한 셀에서 이 값들을 직접 재계산합니다.


### CF 이웃을 남기는 조건

```python
def _terms(self, u: int, columns: np.ndarray) -> tuple:
    # 행은 목표 영화, 열은 유사도순 후보 이웃. 두 축을 같은 집계식에 연결한다.
    if self.axis == 'user':
        order = np.broadcast_to(self._order[u], (len(columns), len(self._order[u])))
        ratings = self._ratings[order, columns[:, None]]
        sims = np.broadcast_to(self._sim[u, self._order[u]], ratings.shape)
        common = np.broadcast_to(self._common[u, self._order[u]], ratings.shape)
        reference = np.broadcast_to(self.user_means_.to_numpy()[self._order[u]], ratings.shape)
        ids = self.rating_matrix_.index.to_numpy()[order]
    else:
        order = self._order[columns]
        ratings = self._ratings[u, order]
        sims = self._sim[columns[:, None], order]
        common = self._common[columns[:, None], order]
        reference = np.zeros_like(ratings)
        ids = self.rating_matrix_.columns.to_numpy()[order]
    keep = np.isfinite(ratings) & (sims > 1e-12) & (common >= self.min_common)
    if self.k:
        keep &= keep.cumsum(axis=1) <= self.k
    weights = np.where(keep, sims, 0)
    values = ratings - reference if self.centered else ratings
    return ratings, values, weights, common, ids, reference
```


### MF 예측 분해

```python
def explain(self, user_id, movie_id) -> dict:
    """ID 두 개의 예측을 평균/두 편향/내적 기여도로 분해한다.

    알려지지 않은 ID의 벡터/편향은 0. 알려진 쪽 편향은 유지한다.
    known_user/known_item으로 cold start를 표시한다. clipping 없음.
    fit 전에는 ValueError. 입력/학습 상태를 수정하지 않는다.
    """
    if not hasattr(self,'P_'): raise ValueError('먼저 fit(ratings)를 실행하세요.')
    u=self.user_index_.get(user_id); i=self.item_index_.get(movie_id)
    bu=float(self.user_bias_[u]) if u is not None else 0.
    bi=float(self.item_bias_[i]) if i is not None else 0.
    contributions=self.P_[u]*self.Q_[i] if u is not None and i is not None else np.zeros(int(self.n_factors))
    interaction=float(contributions.sum())
    return dict(mean=self.mean_,user_bias=bu,item_bias=bi,interaction=interaction,
                prediction=self.mean_+bu+bi+interaction,known_user=u is not None,
                known_item=i is not None,contributions=contributions.copy())
```


### MF SGD 갱신 루프

```python
for epoch in range(1,int(self.epochs)+1):
    for row in rng.permutation(len(rows)):
        a,b=u[row],i[row]
        # copy: 두 벡터는 동일한 갱신 전 상태의 기울기를 사용한다.
        p_old=self.P_[a].copy(); q_old=self.Q_[b].copy()
        e=y[row]-(self.mean_+self.user_bias_[a]+self.item_bias_[b]+p_old@q_old)
        self.P_[a] += self.learning_rate*(e*q_old-self.regularization*p_old)
        self.Q_[b] += self.learning_rate*(e*p_old-self.regularization*q_old)
        if self.use_bias:
            self.user_bias_[a] += self.learning_rate*(e-self.regularization*self.user_bias_[a])
            self.item_bias_[b] += self.learning_rate*(e-self.regularization*self.item_bias_[b])
    record(epoch)
```


### 출처

- 임일, 『AI 에이전트를 위한 개인화 추천 알고리즘』, 청람, 2025, 10장.
- [GroupLens MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)
- [Robin Burke, Hybrid Recommender Systems](https://link.springer.com/article/10.1023/A:1021240730564)
